# Element Creation Examples

In this notebook, we will show some examples for how to create elements as trees.
We will try to start with the thought process, and walk you through the code as we create the desired model.

In [20]:
from coremaker.elements.box import BoxTree, ExcludeFrame, SplitBox, FrameBox, excludeframe_to_framebox
from coremaker.elements.assembly import singular_root_construction
from coremaker.elements.cylindrically_symmetric import AnnulusTree, ChunkedAnnulusTree, ChunkedCylinderTree, CylinderTree
from coremaker.materials.mixture import Mixture
from coremaker.materials.aluminium import al1050, al6061
from coremaker.materials.zirconium import zircalloy_4
from coremaker.materials.absorbers import hafnium, aic, b4c
from coremaker.materials.water import make_light_water
from coremaker.geometries import infiniteGeometry, FiniteCylinder, BareGeometry, Box as BoxGeometry
from coremaker.tree import Tree, Node

# Case 1: MTR-style fuel

We start with the [specification of the OPAL reactor fuel](https://apo.ansto.gov.au/server/api/core/bitstreams/cbb082fd-3e09-44e5-b6e1-ea1e0a122a8f/content), found on pages 3-4.
We show just the figure from those pages, to remind readers what a fuel plate looks like in general.

![OPAL fuel](OPALfuel.png)

The full specification can be a bit tedious to read, but readers who care to do so may open that link and look at the tables while reading this example.

## Breaking the object down
Our first order of business is to deconstruct the object into its underlying components.
In this process, we want to say at each stage if things sit next to each other, or within one another. These correspond to inclusive and exclusive relations between the progeny nodes and their parent node. These concepts are covered in detail in the [Elements](elements.rst) section of the reference guide.

For our MTR fuel assembly, we start with the outmost shape, which is a 81.5mmx81.5mmx1045mm virtual box, or we could imagine it to be within an infinite pool of water instead.
We notice that we can split it into two regions, the `fuel plate` region and the so-called `end box` region.
These sit one atop the other, where the `end box` region is 145mm tall.
In a real setting, outside the end box we would have some free room between this leg and the grid, followed by the core grid material on top of which the fuel rods sit.
For our simplified model, we will assume that outside the `end box` we have light water, and in the `fuel plate` region anything outside the defined geometry is filled with light water at room temperature as well.

We can now break down each of these, and once one of them reaches its simplest shape, we will model it.

### The End Box
The end box is made up of a corner-cut 69mmx69mm square made up of aluminium-6061, where the corners are cut at 8mm isosceles right angle triangles, set **within** our end box region full of water.
**Within** this corner-cut square, we have a cylinder full of water with radius 60mm that is concentric with the square.

Thus, our `end box` tree would be:
    
```mermaid
flowchart TD

A[Infinite Water] -->|External| B[Cut Square]
B -->|External| C[Water Cylinder]
```

We could build that as is, which has the benefit that it follows the actual shape of the components in the model. However, this introduces some complication, since we need to model the cut corners. These make that box region more complicated and has more surfaces, which can sometimes degrade performance on some codes.
Therefore, we are faced with a choice. We can have this simple tree model where the middle node has a complex geometry, or we can build a more complex tree with simpler geometries. We could define this tree instead:

```mermaid
flowchart TD

A[Infinite Water] -->|External| B[69mm Square]
B -->|External| C1[4x 8mmx8mm Isosceles right angle triangles]
B -->|External| C2[Water Cylinder]
```

This construction has more components which will lead to more Cells in an OpenMC-adapted model, but each of which has less surfaces.
It also uses more established geometry types on our end, which increases how idiomatic the model is.

Since we are trying to teach the reader, we will show both options.

### Option 1: Smaller tree, complex geometry
Let's start by creating the materials we use. We have Al-6061 ready, so we just need the water.
We will use the tool for light water that we directly support.

In [3]:
water = make_light_water(25.)

Next, we need to make the infinite geometry and the inner cylinder.
The infiniteGeometry was already imported above, and since it is a singleton instance we don't need to do anything for it. Just the cylinder, then.
It is always best to create geometries with origin centers and leave any
shifts to the transformations in the tree, so we will take care of the z-shift of the end box with transformations.
Our system uses cm as the common unit of measurement, so the engineering-focused use of mm has to be converted.

In [10]:
endbox_cyl = FiniteCylinder(
    center=(0, 0, 0),  # Always prefer origin centers and leave translations to the transforms
    radius=60e-1, #60 mm in cm
    length=145e-1,
    axis=(0, 0, 1), # Z-axis
)

We can now make these into the simpler Nodes.

In [15]:
endbox_leg_flow = Node(geometry=endbox_cyl, mixture=water)
pool_node = Node(geometry=infiniteGeometry, mixture=water)

We can now build the complex geometry and its node

In [18]:
BareGeometry?

Init signature:
BareGeometry(
    _surfaces: Sequence[coremaker.protocols.surface.Surface],
    _volume: float | None = None,
) -> None
Docstring:      A geometry where the only known things are the external surfaces.
File:           ~/workspace/gitkurim/coremaker/coremaker/geometries/bare.py
Type:           type
Subclasses:     

In [22]:
endbox_full = BoxGeometry(center=(0, 0, 0),
                          dimensions=(69e-1, 69e-1, 145e-1))

In [ ]:
d = 
corners = Plane(1, 1, 0, d)

In [23]:
endbox_cut_corners = BareGeometry(
    surfaces=list(endbox_full.surfaces) + corners,
    volume=endbox_full.volume - 4 * triangle_volume)

NameError: name 'corners' is not defined